# 第8回 課題：解答例

## 問1. Dockerfile の主要命令

Python 3.11 ベースで、`requirements.txt` の依存をインストールし、`main.py` を `uvicorn` で起動する Dockerfile を作ります。**レイヤキャッシュを意識した順序**にしてください。

```dockerfile
# ベースイメージ
____ python:3.11-slim

WORKDIR /app

# 依存ライブラリの定義だけを先にコピー
____ requirements.txt .
____ pip install --no-cache-dir -r requirements.txt

# アプリ本体は依存より後にコピー
____ main.py .

EXPOSE 8000

# コンテナ起動コマンド
____ ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

下のセルに、5つの `____` を埋めた**完全な Dockerfile** を文字列として記入してください（読みやすさのため triple-quoted で OK）。

In [ ]:
answer_dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''
print(answer_dockerfile)

**解説**:
- **`requirements.txt` を先に COPY して `pip install` する理由** は **レイヤキャッシュ**。アプリコードを変更しても依存ファイルが変わらなければ、`pip install` レイヤがキャッシュから再利用されてビルドが秒で終わる。逆順だとコード変更のたびに数分待たされる。
- `0.0.0.0` にバインドしないとコンテナ外から到達できない。`127.0.0.1` だと「コンテナ内からしか見えない」状態になる。
- `EXPOSE` は**メタ情報の宣言**。実際にホストへポートを公開するのは `docker run -p` 側の役割。


## 問2. `docker run` のポートフォワード

ビルド済みの `mnist-api:latest` を、**ホストの 9000 番**にコンテナの 8000 番を繋いだ状態で、バックグラウンド起動するコマンドを書きます。

```bash
# ホストのポート9000を、コンテナの8000に繋ぐ
docker run ____ --rm --name mnist-api ____ ____ mnist-api:latest
```

下のセルに、3つの `____` を埋めた**完全な docker run コマンド**を文字列として記入してください。

**ポイント**
- `gr.Image(type='pil')` で PIL Image として受け取る（NumPy が良い場合は `'numpy'`、ファイルパスなら `'filepath'`）
- `gr.Label` は `dict[str, float]` を渡すと上位N件を棒グラフで表示してくれる。多クラス分類の標準。

In [ ]:
answer_docker_run = "docker run -d --rm --name mnist-api -p 9000:8000 mnist-api:latest"
print(answer_docker_run)

## 問3: GitHub Actions ワークフロー

In [ ]:
workflow = '''
name: CI

on:                              # TODO_6: ワークフロートリガの親キー
  push:
    branches: ["main"]

jobs:
  test:
    runs-on: ubuntu-latest       # TODO_7: 最新の Ubuntu runner
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install
        run: pip install -r app/requirements.txt pytest httpx
      - name: Test
        run: pytest app/tests -v # TODO_8: シェルコマンドを書くキー
'''
print(workflow)

**ポイント**
- `on:` がトリガ。`push` `pull_request` `schedule` `workflow_dispatch` などをぶら下げる
- `runs-on:` で実行マシン（`ubuntu-latest` `windows-latest` `macos-latest` など）
- ステップは `uses:`（既製アクション呼び出し）または `run:`（シェル実行）

## 問4: K8s Deployment マニフェスト

In [ ]:
manifest = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mnist-app
spec:
  replicas: 3                    # TODO_3: 3個動かす
  selector:
    matchLabels:
      app: mnist-app
  template:
    metadata:
      labels:
        app: mnist-app
    spec:
      containers:
        - name: mnist-app
          image: mnist-app:v2    # TODO_4: ローカルイメージ名:タグ
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000  # TODO_5: uvicorn が listen するポート
'''
print(manifest)

**ポイント**
- `replicas: 3` で「3個動いてほしい」を宣言。1個落ちても K8s が自動で再生成する
- `imagePullPolicy: IfNotPresent` がローカルイメージ参照のキモ。`Always` だとレジストリを見に行ってしまう
- `containerPort` は **コンテナ内**のポート。Service の `targetPort` がここを指す

## 問5: PSI 計算

In [ ]:
import numpy as np

def calc_psi(reference, current, bins=10, eps=1e-6):
    quantiles = np.linspace(0, 1, bins + 1)
    bin_edges = np.quantile(reference, quantiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)

    p_ref = ref_counts / ref_counts.sum() + eps
    p_cur = cur_counts / cur_counts.sum() + eps

    # TODO_9: PSI = Σ (p_cur - p_ref) * ln(p_cur / p_ref)
    psi = np.sum((p_cur - p_ref) * np.log(p_cur / p_ref))
    return float(psi)

rng = np.random.default_rng(0)
ref = rng.normal(50, 10, 5000)
cur = rng.normal(55, 10, 5000)
print(f'PSI = {calc_psi(ref, cur):.4f}')

**ポイント**
- 「2分布の bin ごとの割合差」を「対数比」で重み付けして合計
- 0.1 未満 / 0.1〜0.2 / 0.2以上 が経験的しきい値
- `eps` を足しているのは log(0) や 0 割を避けるため

## 問6: しきい値判定

In [ ]:
def judge(psi: float) -> str:
    if psi < 0.1:
        return 'stable'
    if psi < 0.2:
        return 'warning'
    return 'drift'

for v in [0.05, 0.15, 0.30]:
    print(v, '->', judge(v))

## 問7: 記述問題の解答例

### Q7-1（解答例）
**コードを変えていないのにモデルの精度が落ちる現象＝データドリフト（あるいはコンセプトドリフト）。**

検知手法の例：**KS検定（Kolmogorov-Smirnov 検定）**。学習時の分布と現在の分布を、**累積分布関数（CDF）の最大乖離**で比較し、p値が小さければ「分布は同じ」という帰無仮説を棄却して**ドリフトあり**と判定する。

別解：**PSI（Population Stability Index）**。等頻度ビンで分布を区切り、各 bin の割合差を対数比で重み付け合計する。0.2 以上で「大きなシフト」と判断する経験的しきい値が広く使われる。

### Q7-2（解答例）
**4ステップの実運用フロー**
1. **記録**：本番の入力分布の統計量（PSI / KS 統計量）を時系列に保存（例：Prometheus、BigQuery）
2. **可視化**：ダッシュボード（例：Grafana）に時系列でプロットし、傾向を観察できるようにする
3. **アラート**：しきい値を超えたら Slack / PagerDuty に通知。連続N回続いた場合に発火させると誤報を減らせる
4. **再学習トリガー**：CT パイプラインを起動し、新データで学習 → 評価 → モデルレジストリ登録 → 自動デプロイ

### Q7-3（解答例）
**Kubernetes が解決する問題（3点）**
1. **自己修復**：単一 `docker run` はコンテナが落ちたら手動で再起動が必要。K8s は Deployment が「N個動かす」を維持するため、落ちても自動で再生成される
2. **スケーラビリティ**：負荷が増えても `docker run` を増やすには手動オペレーションが必要。K8s は HPA で CPU/メモリ等の指標に応じて Pod 数を自動増減できる
3. **負荷分散とサービスディスカバリ**：複数のコンテナを立てても、リクエストの振り分けは別途仕組みが必要。K8s は Service が固定アドレスとして複数 Pod に負荷分散する

他にも：宣言的構成・ローリングアップデート・ノード障害時の Pod 再配置・GPU リソース管理など、許容できる解答は多い。